# GPU Experiment 2A: Primary Natural-Completion Evaluation (Baseline & Linear Decay)

This split notebook evaluates schedules: **baseline, decay** under un-truncated natural completion ($N_{\text{test}}=500$, `max_new_tokens=800`).


In [1]:
!pip install -q evaluate bert_score bitsandbytes accelerate transformers rouge_score

import os, sys, json, time, math, torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evaluate

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device 0: {torch.cuda.get_device_name(0)}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.5 MB/s eta 0:00:00
PyTorch Version: 2.10.0+cu128
CUDA Available: True
GPU Device 0: Tesla T4


In [2]:
possible_paths = [
    '/kaggle/input/datasets/trungkiennnn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese-medical-halueval-15k/vietnamese_medical_halueval_15k_specialized.json',
    'e:/Paper_Steering_VN_15K/data/vietnamese_medical_halueval_15k_specialized.json',
    './data/vietnamese_medical_halueval_15k_specialized.json'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

with open(data_path, 'r', encoding='utf-8') as f:
    full_dataset = json.load(f)

test_data = full_dataset[-500:]
train_pool = full_dataset[:-2205]

model_id = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
model.eval()
bertscore = evaluate.load("bertscore")
print("✅ Model & BERTScore loaded!")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model & BERTScore loaded!


In [3]:
pos_acts, neg_acts = [], []
for item in train_pool[:300]:
    q = item['question']
    pos_ans = item.get('right_answer', item.get('positive_answer'))
    neg_ans = item['hallucinated_answer']
    
    t_pos = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{pos_ans}"
    t_neg = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{neg_ans}"
    
    with torch.no_grad():
        inp_pos = tokenizer(t_pos, return_tensors="pt").to(model.device)
        out_pos = model(inp_pos.input_ids, output_hidden_states=True)
        pos_acts.append(out_pos.hidden_states[8][0, -1, :].detach().cpu())
        
        inp_neg = tokenizer(t_neg, return_tensors="pt").to(model.device)
        out_neg = model(inp_neg.input_ids, output_hidden_states=True)
        neg_acts.append(out_neg.hidden_states[8][0, -1, :].detach().cpu())

v_diff = torch.stack(pos_acts).mean(dim=0) - torch.stack(neg_acts).mean(dim=0)
v_steer = v_diff / v_diff.norm(p=2)
print("✅ Vector v_steer extracted!")


✅ Vector v_steer extracted!


In [4]:
def make_safe_hook(schedule_type="decay", alpha_0=18.0, K=16):
    step_counter = 0
    def hook_fn(module, input_tensor, output_tensor):
        nonlocal step_counter
        step_counter += 1
        
        if schedule_type == "continuous":
            alpha_t = alpha_0
        elif schedule_type == "cutoff":
            alpha_t = alpha_0 if step_counter <= K else 0.0
        elif schedule_type == "decay":
            alpha_t = alpha_0 * (1.0 - (step_counter - 1) / K) if 1 <= step_counter <= K else 0.0
        else:
            alpha_t = 0.0
            
        if alpha_t != 0.0:
            if isinstance(output_tensor, tuple):
                cur_tensor = output_tensor[0]
                v_curr = v_steer.to(device=cur_tensor.device, dtype=cur_tensor.dtype)
                modified = cur_tensor + alpha_t * v_curr
                return (modified,) + output_tensor[1:]
            else:
                v_curr = v_steer.to(device=output_tensor.device, dtype=output_tensor.dtype)
                return output_tensor + alpha_t * v_curr
        return output_tensor
    return hook_fn


In [5]:
target_layer_module = model.model.layers[8]
schedules = ["baseline", "decay"]
results_800 = {}

def compute_rep4(text):
    tokens = text.lower().split()
    if len(tokens) < 4: return 0.0
    ngrams = [tuple(tokens[i:i+4]) for i in range(len(tokens)-3)]
    return (1.0 - len(set(ngrams)) / len(ngrams)) * 100.0

for s in schedules:
    print(f"🚀 Running 800-Token Evaluation for Schedule: {s}...")
    gen_texts, ref_texts, hal_texts = [], [], []
    eos_hits, rep4_list, lengths = [], [], []
    
    for item in tqdm(test_data, desc=f"Evaluating {s} (800 tokens)"):
        prompt = f"<|im_start|>user\n{item['question']}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        p_len = inputs.input_ids.shape[1]
        
        if s != "baseline":
            hook_handle = target_layer_module.register_forward_hook(make_safe_hook(schedule_type=s))
            
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=800, do_sample=False, pad_token_id=tokenizer.pad_token_id)
            
        if s != "baseline":
            hook_handle.remove()
            
        gen_toks = out[0][p_len:]
        gen_text = tokenizer.decode(gen_toks, skip_special_tokens=True)
        gen_texts.append(gen_text)
        ref_texts.append(item.get('right_answer', item.get('positive_answer')))
        hal_texts.append(item['hallucinated_answer'])
        
        eos_hits.append(1 if tokenizer.eos_token_id in gen_toks else 0)
        rep4_list.append(compute_rep4(gen_text))
        lengths.append(len(gen_toks))
        
    bs_ref = bertscore.compute(predictions=gen_texts, references=ref_texts, model_type="bert-base-multilingual-cased")['f1']
    bs_hal = bertscore.compute(predictions=gen_texts, references=hal_texts, model_type="bert-base-multilingual-cased")['f1']
    acc = sum(1 for r, h in zip(bs_ref, bs_hal) if r > h) / len(test_data) * 100.0
    
    results_800[s] = {
        "accuracy": acc,
        "bertscore_f1": float(np.mean(bs_ref)),
        "eos_hit_rate": float(np.mean(eos_hits) * 100.0),
        "mean_rep4": float(np.mean(rep4_list)),
        "mean_output_length": float(np.mean(lengths))
    }
    print(f"Result [{s:10s}]: Acc={acc:.2f}%, BERTScore={np.mean(bs_ref):.4f}, EOS={results_800[s]['eos_hit_rate']:.1f}%, Rep4={results_800[s]['mean_rep4']:.2f}%")


🚀 Running 800-Token Evaluation for Schedule: baseline...



Evaluating baseline (800 tokens):   0%|          | 0/500 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Evaluating baseline (800 tokens): 100%|██████████| 500/500 [2:35:56<00:00, 18.71s/it]


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Result [baseline  ]: Acc=65.40%, BERTScore=0.6779, EOS=100.0%, Rep4=4.10%
🚀 Running 800-Token Evaluation for Schedule: decay...



Evaluating decay (800 tokens): 100%|██████████| 500/500 [3:18:42<00:00, 23.84s/it]


Result [decay     ]: Acc=67.80%, BERTScore=0.6732, EOS=99.8%, Rep4=5.37%


In [6]:
with open("natural_completion_800_a_results.json", "w", encoding="utf-8") as f:
    json.dump(results_800, f, indent=2, ensure_ascii=False)

print("✅ Saved natural_completion_800_a_results.json successfully!")
print(json.dumps(results_800, indent=2))


✅ Saved natural_completion_800_a_results.json successfully!
{
  "baseline": {
    "accuracy": 65.4,
    "bertscore_f1": 0.6779007887840272,
    "eos_hit_rate": 100.0,
    "mean_rep4": 4.0957860166222755,
    "mean_output_length": 248.678
  },
  "decay": {
    "accuracy": 67.80000000000001,
    "bertscore_f1": 0.6731799179315567,
    "eos_hit_rate": 99.8,
    "mean_rep4": 5.369567990853289,
    "mean_output_length": 317.17
  }
}
